# 03 – Xử lý, kiểm chứng và xuất sản phẩm (Bước 3–5)

**Bước 3:** Xử lý theo đúng thứ tự SOP  
**Bước 4:** Kiểm chứng bằng assert  
**Bước 5:** Xuất Parquet, data dictionary, quality report

In [ ]:
import sys
import pickle
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import config as cfg
from parsers import load_all_intermagnet
from pipeline import run_pipeline, run_assertions
from quality_checks import run_all_checks
from export import export_parquet, write_data_dictionary, write_quality_report

print('Setup xong.')

In [ ]:
# Nạp QC results từ Bước 2
qc_path = cfg.DATA_INTERIM / 'qc_before.pkl'
if qc_path.exists():
    with open(qc_path, 'rb') as f:
        qc_results_before = pickle.load(f)
    print(f'✅ Nạp QC before: {list(qc_results_before.keys())}')
else:
    print('⚠  Không tìm thấy qc_before.pkl – chạy notebook 02 trước')
    qc_results_before = {}

In [ ]:
# ── Bước 3: Chạy pipeline xử lý ──────────────────────────────────────────
clean_dfs    = {}
proc_logs    = {}
station_meta = {}

for station in cfg.STATIONS:
    try:
        df_raw, meta = load_all_intermagnet(station)
        station_meta[station] = meta
        
        df_clean, log = run_pipeline(
            df_raw,
            station=station,
            sentinel_values=meta.get('sentinel_values', cfg.SENTINEL_VALUES)
        )
        clean_dfs[station] = df_clean
        proc_logs[station] = log
    except FileNotFoundError as e:
        print(f'⚠  {e}')

In [ ]:
# ── Bước 4: Kiểm chứng ────────────────────────────────────────────────────
for station, df_clean in clean_dfs.items():
    qc_before = qc_results_before.get(station, {})
    run_assertions(
        df_clean,
        station=station,
        expected_rows=qc_before.get('gaps', {}).get('expected_points'),
        qc_before=qc_before
    )

In [ ]:
# ── Bước 5a: Xuất Parquet ─────────────────────────────────────────────────
for station, df_clean in clean_dfs.items():
    export_parquet(
        df_clean,
        source='intermagnet',
        station=station,
        freq='1min',
        also_csv=False  # Đổi thành True nếu muốn thêm CSV
    )

In [ ]:
# ── Bước 5b: QC sau xử lý (để so sánh trước/sau) ─────────────────────────
qc_results_after = {}
for station, df_clean in clean_dfs.items():
    result_after = run_all_checks(df_clean, station=station, label='after')
    qc_results_after[station] = result_after

In [ ]:
# ── Bước 5c: Xuất Data Dictionary và Quality Report ───────────────────────
for station in clean_dfs:
    df_clean = clean_dfs[station]
    meta     = station_meta.get(station)
    
    print(f'\n── {station} ──')
    
    # Data Dictionary (lấy schema từ df clean)
    write_data_dictionary(df_clean, meta=meta)
    
    # Quality Report
    write_quality_report(
        qc_before=qc_results_before.get(station, {}),
        qc_after=qc_results_after.get(station, {}),
        processing_log=proc_logs.get(station, {}),
        meta=meta
    )

print('\n✅ Tất cả sản phẩm đã xuất xong!')
print(f'   data/clean/   : {list((cfg.DATA_CLEAN).glob("*.parquet"))}')
print(f'   docs/         : {list(cfg.DOCS_DIR.glob("*.md"))}')

In [ ]:
# ── Kiểm chứng tái lập: chạy lại pipeline và so sánh ─────────────────────
print('Kiểm chứng tái lập...')
for station in cfg.STATIONS:
    if station not in clean_dfs:
        continue
    df_raw2, meta2 = load_all_intermagnet(station)
    df_clean2, _ = run_pipeline(
        df_raw2, station=station,
        sentinel_values=meta2.get('sentinel_values', cfg.SENTINEL_VALUES)
    )
    # So sánh giá trị đo lường
    meas = [c for c in df_clean2.columns if c.endswith('_nt')]
    for col in meas:
        assert df_clean2[col].equals(clean_dfs[station][col]), f'❌ Không tái lập: {station}/{col}'
    print(f'  ✅ {station}: tái lập thành công')

print('\n🎉 Pipeline hoàn toàn tái lập được!')